# Detect → anonymise → re-detect (10s clip)

Merges the two existing notebooks into one A/B run:

1. **Trim** the source to a 10s clip (re-encoded, never stream-copied).
2. **Detect** faces with RetinaFace on the *original* clip; render an overlay video.
3. **Anonymise** it with Pipeline A (SCRFD → IoU track → pixelate → ffmpeg render).
4. **Detect** again with RetinaFace on the *anonymised* video; render a second overlay.
5. **Compare** detection statistics from both passes, then run the independent
   verification gate.

Merging fixes a real bug. `RetinaFaceTest.ipynb` pointed its `VIDEO` at a
*Pipeline A output* (`notebooks/out/anon-*.mp4`), so it was detecting faces in
already-anonymised footage. Here the two passes are explicit about which file
they read.

## How to read the numbers

A drop in detections after anonymisation is **expected, and is not proof that
anonymisation worked**. RetinaFace failing to parse a pixelated face is weak
evidence — it would equally fail on a badly-lit but perfectly visible face. The
before/after count is a *sanity signal*: if it does not drop, something is wrong.

The real leak test is cell 19: RetinaFace hitting a face in the **anonymised**
video that Pipeline A never tracked. SCRFD and RetinaFace are independent model
families, which is exactly what makes that cross-check meaningful.

This notebook never edits `common/config.py`; `PIPELINE_VERSION` is read-only here.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# Must be set BEFORE anything imports the retinaface verifier.
os.environ.setdefault("DEEPFACE_BACKEND_ENGINE", "pytorch")


def find_root():
    """Locate the repo root (the dir containing common/config.py).

    Tries the cwd, then the notebook's own directory, then sys.path entries, so
    the notebook works from any launch directory.
    """
    starts = [Path.cwd()]
    try:
        nb_path = get_ipython().get_parent()["metadata"].get("path")
        if nb_path:
            starts.append(Path(nb_path).resolve().parent)
    except Exception:
        pass
    for entry in sys.path:
        if entry:
            starts.append(Path(entry))

    seen = set()
    for start in starts:
        try:
            here = start.resolve()
        except OSError:
            continue
        for cand in [here] + list(here.parents):
            if cand in seen:
                continue
            seen.add(cand)
            if (cand / "common" / "config.py").is_file():
                return cand
    raise RuntimeError(
        "could not locate the repo root (no common/config.py found). "
        "Launch Jupyter from the repository root, or add the repo to PYTHONPATH.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import matplotlib
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Video, display

RESULTS = []
_MODEL_CACHE = {}


def check(step, ok, detail=""):
    """Record and print one PASS/FAIL line."""
    status = "PASS" if ok else "FAIL"
    print("[%s] %s%s" % (status, step, (" -- " + detail) if detail else ""),
          flush=True)
    RESULTS.append((step, status, detail))
    return bool(ok)


def cache_model(key, builder):
    """Build heavy models at most once per kernel session."""
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = builder()
    return _MODEL_CACHE[key]


def banner(title):
    print("\n" + "=" * 72)
    print(title)
banner("CELL 1  setup + environment preflight")
print("sys.executable :", sys.executable)
print("ROOT           :", ROOT)

# Homebrew (and friends) install outside the PATH a GUI-launched Jupyter
# inherits, so probe common locations and repair PATH before anything shells
# out to ffmpeg/ffprobe.
EXTRA_BIN_DIRS = ["/opt/homebrew/bin", "/usr/local/bin", "/opt/local/bin",
                  "/usr/bin", "/bin", "/usr/local/sbin", "/usr/sbin"]


def ensure_on_path(names):
    """Return {name: path}; append any missing tool's dir to PATH."""
    found, added = {}, []
    for name in names:
        path = shutil.which(name)
        if path:
            found[name] = path
            continue
        for directory in EXTRA_BIN_DIRS:
            candidate = os.path.join(directory, name)
            if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
                found[name] = candidate
                if directory not in added:
                    added.append(directory)
                break
    if added:
        os.environ["PATH"] = os.pathsep.join(added + [os.environ.get("PATH", "")])
        print("added to PATH   :", added)
    return found


TOOLS = ensure_on_path(["ffmpeg", "ffprobe"])
ffmpeg, ffprobe = TOOLS.get("ffmpeg"), TOOLS.get("ffprobe")
print("ffmpeg         :", ffmpeg)
print("ffprobe        :", ffprobe)
TOOLS_OK = bool(ffmpeg and ffprobe)
check("env: ffmpeg + ffprobe resolvable", TOOLS_OK,
      "" if TOOLS_OK else "install ffmpeg, or add it to PATH")
if not TOOLS_OK:
    print("\nWARNING: ffmpeg/ffprobe unavailable; every later cell will FAIL.")

weights = {
    "insightface buffalo_l (Pipeline A)":
        (Path.home() / ".insightface" / "models" / "buffalo_l").is_dir(),
    "deepface retinaface.pth (verifier)":
        (Path.home() / ".deepface" / "weights" / "retinaface.pth").is_file(),
}
for name, ok in weights.items():
    print("  weight %-40s %s" % (name, "present" if ok else "MISSING"))
check("env: model weights present", all(weights.values()))

try:
    import torch as _torch_mod

    _threads = os.cpu_count() or 4
    _torch_mod.set_num_threads(_threads)
    print("torch threads   : %d (of %s cpus)"
          % (_torch_mod.get_num_threads(), _threads))
except Exception as exc:
    print("torch thread tuning skipped: %s" % exc)

mods = {}
for name in ("cv2", "numpy", "torch", "insightface", "retinaface"):
    try:
        __import__(name)
        mods[name] = True
    except Exception as exc:
        mods[name] = False
        print("  import %-12s FAILED: %s" % (name, exc))
check("env: imports resolve", all(mods.values()), "%d modules" % len(mods))

from common import config as config_mod

print("\nPIPELINE_VERSION (read-only here):", config_mod.PIPELINE_VERSION)

## PARAMS — the only cell to edit

In [ ]:
# >>> EDIT HERE <<<
INPUT_VIDEO = ROOT / "downloads/face_detection_test.mp4"
CLIP_SECONDS = 10.0          # hard cap; the trim never exceeds this
CLIP_START = 0.0            # seconds into the source to start the clip
CLIP_DIR = ROOT / "notebooks" / "out"
PREVIEW_DIR = ROOT / "previews"

# Stride 1 = detect every frame (final validation). CPU RetinaFace at 1080p
# costs several s/frame here (measured ~8s/frame, so a 300-frame clip is
# ~40 min per sweep; this notebook performs up to three sweeps: pass 1 +
# pass 2 + the cell-10 gate). Default 5 keeps stats honest via the held-box
# mechanism and cuts each sweep ~5x; set 1 for the final validation run.
DET_STRIDE = 5

# Detect at reduced width (0 = full resolution). 640 keeps recall on 1080p
# talking-head footage while cutting inference ~3-4x; boxes are scaled back
# to full resolution before drawing/stats. Set 0 for final validation.
DET_WIDTH = 640

# Pass 2 re-sweeps the anonymised video with identical code. The cell-10 gate
# (run_verify, same model family + threshold, stride-1 coverage) already
# sweeps the anonymised file, so for exploratory runs the pass-2 sweep is
# redundant: False skips it and reuses the gate's numbers. True renders
# previews/det_anon.mp4 (needed for the final validation run).
RUN_PASS2_SWEEP = False

# Defaults to the repo's own verifier threshold so these numbers line up with
# what Pipeline B would report.
SCORE_THRESHOLD = config_mod.DEFAULT_CONFIG["verifier"]["det_threshold"]

RUN_GATE = True             # cell 19: independent verification + decision gate

CLIP_PATH = CLIP_DIR / ("clip_%ds.mp4" % int(CLIP_SECONDS))
ORIG_OVERLAY = PREVIEW_DIR / "det_orig.mp4"
ANON_OVERLAY = PREVIEW_DIR / "det_anon.mp4"

CLIP_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline A takes no overrides by default: it runs on DEFAULT_CONFIG so the
# numbers here describe the shipped configuration, not a tuned one.
USER_CONFIG = {}

banner("CELL 2  params")
print("INPUT_VIDEO    :", INPUT_VIDEO)
print("clip           : %s  [%s -> %s]"
      % (CLIP_PATH.name, CLIP_START, CLIP_START + CLIP_SECONDS))
print("DET_STRIDE     :", DET_STRIDE)
print("DET_WIDTH      :", DET_WIDTH if DET_WIDTH else "full resolution")
print("RUN_PASS2_SWEEP:", RUN_PASS2_SWEEP)
print("SCORE_THRESHOLD: %.2f (matches DEFAULT_CONFIG['verifier'])" % SCORE_THRESHOLD)
print("RUN_GATE       :", RUN_GATE)
print("\nconfig diff vs DEFAULT_CONFIG:", "(none - using defaults)" if not USER_CONFIG
      else USER_CONFIG)
check("params: input video exists", INPUT_VIDEO.is_file(), str(INPUT_VIDEO))
check("params: clip window sane", CLIP_SECONDS > 0 and CLIP_START >= 0,
      "%.1fs from %.1fs" % (CLIP_SECONDS, CLIP_START))

## Trim the source to a 10s clip

Re-encodes rather than stream-copying. `-c copy` on an AV1 source snaps to the
nearest keyframe, which would give an unpredictable duration and frame count;
re-encoding pins the clip to exactly `CLIP_SECONDS` and makes seeking reliable
for the frame grabs in cell 16.

If the source is already shorter than the window, it is copied verbatim.

In [ ]:
import cv2
import numpy as np

from common.video_info import collect_video_info, probe_video

banner("CELL 3  trim source -> %s" % CLIP_PATH.name)


def has_audio(path):
    """True if ffprobe finds an audio stream."""
    proc = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a",
         "-show_entries", "stream=index", "-of", "csv", str(path)],
        capture_output=True, text=True, check=False)
    return proc.returncode == 0 and "stream" in proc.stdout


def build_clip():
    """Produce CLIP_PATH from INPUT_VIDEO. Returns True if it re-encoded."""
    if not INPUT_VIDEO.is_file():
        raise RuntimeError(
            "input video missing: %s. Place the source clip at "
            "downloads/face_detection_test.mp4 (excluded from the "
            "submission via .gitignore) or point INPUT_VIDEO at it." % INPUT_VIDEO)
    if not TOOLS_OK:
        raise RuntimeError("ffmpeg/ffprobe unavailable (see cell 1)")

    src_dur = probe_video(str(INPUT_VIDEO))["duration_seconds"] or 0.0
    src_info = collect_video_info(str(INPUT_VIDEO))
    print("source         : %.2fs  %dx%d @ %.3f fps  (%s)"
          % (src_dur, src_info["width"], src_info["height"],
             src_info["fps"], src_info["codec"]))

    if src_dur > 0 and src_dur <= CLIP_SECONDS + 0.05:
        print("source is already <= %.1fs - copying verbatim" % CLIP_SECONDS)
        shutil.copy2(str(INPUT_VIDEO), str(CLIP_PATH))
        return False

    cmd = ["ffmpeg", "-y", "-v", "error", "-nostdin",
           "-ss", "%.3f" % CLIP_START, "-i", str(INPUT_VIDEO),
           "-t", "%.3f" % CLIP_SECONDS,
           "-map", "0:v:0", "-c:v", "libx264", "-crf", "20",
           "-preset", "veryfast", "-pix_fmt", "yuv420p"]
    if has_audio(INPUT_VIDEO):
        cmd += ["-map", "0:a:0", "-c:a", "aac"]
    cmd += ["-an" if not has_audio(INPUT_VIDEO) else "-shortest", str(CLIP_PATH)]
    proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print("ffmpeg rc=%d %s" % (proc.returncode, proc.stderr.strip()[-300:]))
    if proc.returncode != 0 or not CLIP_PATH.is_file():
        raise RuntimeError("trim failed: %s" % proc.stderr.strip()[-2000:])
    return True


CLIP_REENCODED = build_clip()
_clip_probe = probe_video(str(CLIP_PATH))
# Count only the frames we expect to exist. common/video_info's decoder counts a
# normal end-of-stream read as a decode error, so an uncapped probe always
# reports errors>=1 and would fail the check below for a perfectly healthy clip.
EXPECTED_FRAMES = int(round(_clip_probe["fps"] * _clip_probe["duration_seconds"]))
CLIP_INFO = collect_video_info(str(CLIP_PATH), max_frames=EXPECTED_FRAMES)
print("\nclip           : %.2fs  %dx%d @ %.3f fps  (%s)"
      % (CLIP_INFO["duration_seconds"], CLIP_INFO["width"], CLIP_INFO["height"],
         CLIP_INFO["fps"], CLIP_INFO["codec"]))
print("decoded frames : %s   decode errors: %s"
      % (CLIP_INFO["decoded_frames"], CLIP_INFO["decode_errors"]))
print("has audio      :", has_audio(CLIP_PATH))

CLIP_FRAMES = CLIP_INFO["decoded_frames"]
check("trim: clip exists", CLIP_PATH.is_file(), CLIP_PATH.name)
check("trim: clip decodes", CLIP_FRAMES > 0, "%d frames" % CLIP_FRAMES)
check("trim: duration within %.1fs" % CLIP_SECONDS,
      abs(CLIP_INFO["duration_seconds"] - CLIP_SECONDS) < 0.2,
      "%.2fs" % CLIP_INFO["duration_seconds"])
check("trim: no decode errors", not CLIP_INFO["decode_errors"])

_cap = cv2.VideoCapture(str(CLIP_PATH))
_ok, _frame0 = _cap.read()
_cap.release()
if _ok and _frame0 is not None:
    plt.figure(figsize=(10, 5.6))
    plt.imshow(_frame0[:, :, ::-1])
    plt.axis("off")
    plt.title("CLIP frame 0  (the first frame everything below works from)")
    plt.tight_layout()
    plt.show()
    check("trim: frame 0 decodes", True, "shape=%s" % (_frame0.shape,))

## RetinaFace detection helpers

Two guards worth noting, both learned the hard way:

- **`facial_area` shape.** retina-face returns corner coordinates
  `[x1, y1, x2, y2]` (installed `retinaface==0.0.19`:
  `facial_area = list(face[0:4])` are NMS proposal
  corners). `normalise_area()` passes the list through as corners and clamps
  to the frame -- an earlier revision treated it as `[x, y, w, h]` and drew
  boxes roughly twice the face size. A `dict` branch is kept defensively
  (iterating a dict would raise `TypeError`) but never fires with
  `RetinaFace.detect_faces`. `verify/retinaface_check.py` already treats the
  value as corners and is unaffected.
- **No seeking.** Every helper decodes sequentially. OpenCV frame seeking on
  compressed video can silently land on a different frame, which would put a box
  on the wrong picture.

In [ ]:
from retinaface import RetinaFace

BGR_FRESH = (0, 255, 0)       # green  = detected on this very frame
BGR_HELD = (0, 165, 255)      # amber  = carried forward from an earlier frame
FONT = cv2.FONT_HERSHEY_SIMPLEX

RETINA = cache_model("retinaface", RetinaFace.build_model)
print("retinaface model ready")


def normalise_area(area):
    """retina-face facial_area -> [x1, y1, x2, y2] corners.

    Both the pip-installed retina-face and the (now-removed) local
    `retinaface/` clone returned NMS proposal corners, NOT
    [x, y, w, h] -- see retinaface/retinaface/RetinaFace.py
    (`facial_area = list(face[0:4])`). The dict branch is defensive only;
    detect_faces never returns it.
    """
    if isinstance(area, dict):
        x, y, w, h = (area["x"], area["y"], area["w"], area["h"])
        return [float(x), float(y), float(x) + float(w), float(y) + float(h)]
    x1, y1, x2, y2 = (float(v) for v in area)
    return [float(x1), float(y1), float(x2), float(y2)]


def clamp_box(box, width, height):
    """Clamp [x1, y1, x2, y2] to the frame; None if degenerate."""
    x1 = min(max(0.0, box[0]), float(width - 1))
    y1 = min(max(0.0, box[1]), float(height - 1))
    x2 = min(max(0.0, box[2]), float(width - 1))
    y2 = min(max(0.0, box[3]), float(height - 1))
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2]


def detect_faces(frame_bgr, model, threshold):
    """Detect on one BGR frame -> {key: {score, box, landmarks}}.

    Boxes are full-resolution [x1, y1, x2, y2] corners regardless of
    DET_WIDTH: detection may run on a downscaled copy, but boxes are
    scaled back before clamping so overlays and stats stay comparable.
    """
    det_frame, scale = frame_bgr, 1.0
    if DET_WIDTH and frame_bgr.shape[1] > DET_WIDTH:
        scale = DET_WIDTH / float(frame_bgr.shape[1])
        det_h = max(1, int(round(frame_bgr.shape[0] * scale)))
        det_frame = cv2.resize(frame_bgr, (DET_WIDTH, det_h),
                               interpolation=cv2.INTER_LINEAR)
    raw = RetinaFace.detect_faces(det_frame, threshold=threshold, model=model)
    out = {}
    if not isinstance(raw, dict):
        return out
    full_w, full_h = frame_bgr.shape[1], frame_bgr.shape[0]
    for key, info in raw.items():
        box = normalise_area(info["facial_area"])
        if scale != 1.0:
            box = [v / scale for v in box]
        box = clamp_box(box, full_w, full_h)
        if box is None:
            continue
        scaled_lm = {}
        for n, coords in (info.get("landmarks") or {}).items():
            pts = [float(c) for c in coords]
            if scale != 1.0:
                pts = [c / scale for c in pts]
            scaled_lm[n] = pts
        out[key] = {"score": float(info["score"]), "box": box,
                    "landmarks": scaled_lm}
    return out


def draw_overlay(frame, faces, held, frame_no, total, fps, label):
    """Draw boxes/landmarks and a status bar. Does not mutate the source frame."""
    annotated = frame.copy()
    h, w = annotated.shape[:2]
    color = BGR_HELD if held else BGR_FRESH

    for key, face in faces.items():
        x1, y1, x2, y2 = (int(v) for v in face["box"])
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        text = "%s %.2f%s" % (key, face["score"], " held" if held else "")
        cv2.putText(annotated, text, (x1, max(18, y1 - 8)), FONT, 0.8,
                    color, 2, cv2.LINE_AA)
        for lx, ly in face["landmarks"].values():
            if 0 <= int(lx) < w and 0 <= int(ly) < h:
                cv2.circle(annotated, (int(lx), int(ly)), 4, (0, 0, 255), -1)

    bar = "%s  f%d/%d  t=%.2fs  %d face(s)%s" % (
        label, frame_no, total, frame_no / fps, len(faces),
        "  [HELD]" if held and faces else "")
    cv2.rectangle(annotated, (0, 0), (w, 34), (0, 0, 0), -1)
    cv2.putText(annotated, bar, (8, 23), FONT, 0.6, (255, 255, 255), 1,
                cv2.LINE_AA)
    return annotated
def run_detection_pass(src_path, out_path, label):
    """Detect on one video, write an annotated copy, return statistics.

    Decodes exactly once, so every drawn box provably belongs to the frame it is
    drawn on. Frames between detections carry the last boxes forward (amber,
    labelled `held`) - at stride 1 there are none, but the mechanism keeps the
    statistics honest if DET_STRIDE is raised.
    """
    started = time.time()
    cap = cv2.VideoCapture(str(src_path))
    if not cap.isOpened():
        raise RuntimeError("could not open %s" % src_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    silent = out_path.with_name(out_path.stem + ".silent.mp4")
    writer = cv2.VideoWriter(str(silent), cv2.VideoWriter_fourcc(*"mp4v"),
                             fps, (width, height))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError("could not open intermediate writer")

    per_frame, scores, boxes, held_boxes = [], [], [], []
    held, held_age = None, 0
    frame_no = detected_frames = held_frames = written = 0
    max_carry = max(1, DET_STRIDE - 1)
    infer_seconds, infer_calls = 0.0, 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok or frame is None or frame.size == 0:
                break
            fresh = (frame_no % max(1, DET_STRIDE)) == 0
            if fresh:
                t0 = time.time()
                faces = detect_faces(frame, RETINA, SCORE_THRESHOLD)
                infer_seconds += time.time() - t0
                infer_calls += 1
                held, held_age = faces, 0
                detected_frames += 1
                for face in faces.values():
                    scores.append(face["score"])
                    boxes.append(face["box"])
                if infer_calls % 50 == 0:
                    print("  [%s] frame %d: %.2fs/infer over %d detection frames"
                          % (label, frame_no, infer_seconds / infer_calls,
                             infer_calls), flush=True)
            elif held and held_age < max_carry:
                faces, held_age = held, held_age + 1
                held_frames += 1
                for face in faces.values():
                    held_boxes.append(face["box"])
            else:
                faces = {}

            per_frame.append(len(faces))
            writer.write(draw_overlay(frame, faces, not fresh, frame_no,
                                       1 << 30, fps, label))
            written += 1
            frame_no += 1
    finally:
        cap.release()
        writer.release()

    cmd = ["ffmpeg", "-y", "-v", "error", "-nostdin",
           "-i", str(silent), "-i", str(src_path),
           "-map", "0:v:0", "-c:v", "libx264", "-crf", "20",
           "-preset", "veryfast", "-pix_fmt", "yuv420p",
           "-frames:v", str(written)]
    if has_audio(src_path):
        cmd += ["-map", "1:a:0", "-c:a", "aac"]
    cmd.append(str(out_path))
    proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
    try:
        silent.unlink()
    except OSError:
        pass
    if proc.returncode != 0:
        raise RuntimeError("ffmpeg encode failed: %s" % proc.stderr.strip()[-2000:])

    return {
        "label": label,
        "source": Path(src_path),
        "overlay": out_path,
        "frames": written,
        "fps": fps,
        "width": width,
        "height": height,
        "per_frame_counts": per_frame,
        "detected_frames": detected_frames,
        "held_frames": held_frames,
        "frames_with_faces": sum(1 for c in per_frame if c > 0),
        "total_detections": len(scores),
        "max_in_frame": max(per_frame) if per_frame else 0,
        "mean_per_frame": (sum(per_frame) / float(written)) if written else 0.0,
        "mean_score": (sum(scores) / len(scores)) if scores else 0.0,
        "max_score": max(scores) if scores else 0.0,
        "scores": scores,
        "boxes": boxes,
        "held_boxes": held_boxes,
        "elapsed_seconds": round(time.time() - started, 2),
        "infer_seconds": round(infer_seconds, 2),
        "infer_calls": infer_calls,
        "infer_per_frame": (round(infer_seconds / infer_calls, 3)
                            if infer_calls else 0.0),
        "det_width": DET_WIDTH,
        "stride": DET_STRIDE,
    }

## Pass 1 — detect on the ORIGINAL clip

This is the baseline: how many faces are visible before anything is obscured.
`previews/det_orig.mp4` is the annotated copy.

In [ ]:
banner("CELL 5  pass 1: RetinaFace on the ORIGINAL clip")
print("source :", CLIP_PATH)
print("overlay:", ORIG_OVERLAY)
n_detect = (CLIP_FRAMES + max(1, DET_STRIDE) - 1) // max(1, DET_STRIDE)
print("~%d detection frames (stride %d of %d) at %s; CPU RetinaFace runs"
      " several s/frame, so budget accordingly"
      % (n_detect, DET_STRIDE, CLIP_FRAMES,
         ("downscaled %dpx" % DET_WIDTH) if DET_WIDTH else "full 1080p"),
      flush=True)

ORIG_STATS = run_detection_pass(CLIP_PATH, ORIG_OVERLAY, "ORIGINAL")

print("\nframes            : %d @ %.2f fps  %dx%d"
      % (ORIG_STATS["frames"], ORIG_STATS["fps"],
         ORIG_STATS["width"], ORIG_STATS["height"]))
print("frames with faces : %d" % ORIG_STATS["frames_with_faces"])
print("total detections  : %d" % ORIG_STATS["total_detections"])
print("mean faces/frame  : %.3f  (max %d in one frame)"
      % (ORIG_STATS["mean_per_frame"], ORIG_STATS["max_in_frame"]))
print("scores            : mean %.3f  max %.3f"
      % (ORIG_STATS["mean_score"], ORIG_STATS["max_score"]))
print("elapsed           : %.1fs (infer %.1fs over %d frames = %.2fs/frame)"
      % (ORIG_STATS["elapsed_seconds"], ORIG_STATS["infer_seconds"],
         ORIG_STATS["infer_calls"], ORIG_STATS["infer_per_frame"]))
print("overlay           : %s (%.1f MB)"
      % (ORIG_OVERLAY, ORIG_OVERLAY.stat().st_size / 1e6))

display(Video(str(ORIG_OVERLAY), embed=True))

check("pass 1: overlay written", ORIG_OVERLAY.is_file(), ORIG_OVERLAY.name)
check("pass 1: frame count matches clip",
      ORIG_STATS["frames"] == CLIP_FRAMES,
      "%d vs %d" % (ORIG_STATS["frames"], CLIP_FRAMES))
check("pass 1: faces actually detected", ORIG_STATS["total_detections"] > 0,
      "%d detections" % ORIG_STATS["total_detections"])

## Anonymise with Pipeline A

Calls `run_pipeline_a` directly — the same entry point the API and the job
worker use. Not a notebook-local copy, so these numbers describe the shipped
pipeline rather than a parallel implementation that can drift.

In [ ]:
from pipeline.detect import build_anonymizer_model
from pipeline.run_pipeline_a import run_pipeline_a

banner("CELL 6  anonymise (Pipeline A: SCRFD -> track -> pixelate -> render)")

det_size = tuple(config_mod.canonical_config(USER_CONFIG)["anonymizer"]["det_size"])
detector = cache_model("scrfd-%s" % (det_size,),
                       lambda: build_anonymizer_model(det_size=det_size, ctx_id=-1))

# max_frames is passed deliberately (the clip is already bounded): with
# max_frames=0 the decoder reads to EOF and counts that final read as a decode
# error, which would push a spurious `pipeline_a_decode_errors=1` into the gate.
PIPELINE_A = run_pipeline_a(str(CLIP_PATH), str(CLIP_DIR), USER_CONFIG,
                            max_frames=CLIP_FRAMES, model=detector)
RENDERED = Path(PIPELINE_A["output_path"])

print("output          :", RENDERED)
print("frames decoded  : %s" % PIPELINE_A["input_info"]["decoded_frames"])
print("frames rendered : %s" % PIPELINE_A["rendered_frames"])
print("frames covered  : %s (blur boxes present)" % PIPELINE_A["covered_frames"])
print("tracks          : %s" % PIPELINE_A["num_tracks"])
print("bridged gaps    : %s (%s inferred frames)"
      % (PIPELINE_A["num_bridged_gaps"], PIPELINE_A["bridged_frames"]))
print("elapsed         : %ss" % PIPELINE_A["elapsed_seconds"])

check("pipeline A: output exists", RENDERED.is_file(), RENDERED.name)
check("pipeline A: rendered == decoded",
      PIPELINE_A["rendered_frames"] == PIPELINE_A["input_info"]["decoded_frames"],
      "%s vs %s" % (PIPELINE_A["rendered_frames"],
                    PIPELINE_A["input_info"]["decoded_frames"]))
check("pipeline A: blurred at least some frames", PIPELINE_A["covered_frames"] > 0,
      "%s frames" % PIPELINE_A["covered_frames"])

## Pass 2 — detect on the ANONYMISED video

Identical code, identical model instance, identical threshold — the only thing
that changed is which file is being read. Any difference in the numbers is
therefore attributable to the anonymisation, not to a changed method.

Skippable: with `RUN_PASS2_SWEEP = False` (default) this cell only records
`ANON_STATS = None`, and the verification gate's sweep of the same file
stands in for the comparison tables. Same model family, same threshold,
stricter stride-1 coverage — and one fewer ~40-minute sweep.

In [ ]:
banner("CELL 7  pass 2: RetinaFace on the ANONYMISED clip")
print("source :", RENDERED)
print("overlay:", ANON_OVERLAY)
print("stride         :", DET_STRIDE,
      ("  width %dpx" % DET_WIDTH) if DET_WIDTH else "  full resolution")
ANON_STATS = None
if RUN_PASS2_SWEEP:
    ANON_STATS = run_detection_pass(RENDERED, ANON_OVERLAY, "ANONYMISED")
else:
    print("SKIPPED - RUN_PASS2_SWEEP=False. The verification gate sweeps")
    print("the same anonymised file with the same model family + threshold,")
    print("so its numbers stand in below. Set RUN_PASS2_SWEEP=True for the")
    print("final validation run (also renders %s)." % ANON_OVERLAY.name)
    check("pass 2: sweep skipped (gate stands in)", True,
          "RUN_PASS2_SWEEP=False")
if ANON_STATS is not None:
    print("\nframes            : %d @ %.2f fps  %dx%d"
          % (ANON_STATS["frames"], ANON_STATS["fps"],
             ANON_STATS["width"], ANON_STATS["height"]))
    print("frames with faces : %d" % ANON_STATS["frames_with_faces"])
    print("total detections  : %d" % ANON_STATS["total_detections"])
    print("mean faces/frame  : %.3f  (max %d in one frame)"
          % (ANON_STATS["mean_per_frame"], ANON_STATS["max_in_frame"]))
    print("scores            : mean %.3f  max %.3f"
          % (ANON_STATS["mean_score"], ANON_STATS["max_score"]))
    if ANON_STATS.get("infer_calls"):
        print("elapsed           : %.1fs (infer %.1fs over %d frames = "
              "%.2fs/frame)"
              % (ANON_STATS["elapsed_seconds"], ANON_STATS["infer_seconds"],
                 ANON_STATS["infer_calls"], ANON_STATS["infer_per_frame"]))
    else:
        print("elapsed           : %.1fs" % ANON_STATS["elapsed_seconds"])
    print("overlay           : %s (%.1f MB)"
          % (ANON_OVERLAY, ANON_OVERLAY.stat().st_size / 1e6))
    display(Video(str(ANON_OVERLAY), embed=True))
    check("pass 2: overlay written", ANON_OVERLAY.is_file(), ANON_OVERLAY.name)
    check("pass 2: frame count matches pipeline A output",
          ANON_STATS["frames"] == PIPELINE_A["rendered_frames"],
          "%d vs %d" % (ANON_STATS["frames"], PIPELINE_A["rendered_frames"]))
    check("pass 2: same geometry as pass 1",
          (ANON_STATS["width"], ANON_STATS["height"]) ==
          (ORIG_STATS["width"], ORIG_STATS["height"]),
          "%dx%d" % (ANON_STATS["width"], ANON_STATS["height"]))

## Independent verification gate

This is the actual leak test. `run_verify` re-reads the **rendered file from
disk** — not Pipeline A's in-memory frames — and runs RetinaFace over a
predetermined plan that covers every output frame plus windows around track
boundaries. Any face it still finds is a flag.

Why this is stronger than the counts in cell 9: Pipeline A detects with SCRFD
and this verifies with RetinaFace. Two independent model families, so a flag is
not just the anonymiser confirming its own work.

`decide()` is review-by-default. `NEEDS_REVIEW` is the expected outcome; `SAFE`
has to be earned by every check passing.

In [ ]:
from jobs import decision as decision_mod
from verify.retinaface_check import build_verifier_model, run_verify

banner("CELL 10  verification gate")

if not RUN_GATE:
    print("SKIPPED - set RUN_GATE = True in the PARAMS cell to run this step.")
    check("gate: optional verification", True, "skipped (RUN_GATE=False)")
else:
    CROPS_DIR = CLIP_DIR / "crops"
    CROPS_DIR.mkdir(parents=True, exist_ok=True)
    verifier = cache_model("retinaface", build_verifier_model)

    VERIFICATION = run_verify(str(RENDERED), PIPELINE_A["tracks"], USER_CONFIG,
                              model=verifier, crops_dir=str(CROPS_DIR),
                              max_frames=CLIP_FRAMES)

    print("threshold      : %s" % VERIFICATION["threshold"])
    print("stride         : %s" % VERIFICATION["stride"])
    print("coverage       : %.4f (%s/%s planned frames)"
          % (VERIFICATION["coverage"], VERIFICATION["covered_frames"],
             VERIFICATION["planned_frames"]))
    print("decode errors  : %s" % VERIFICATION["decode_errors"])
    print("verifier hits  : %s" % VERIFICATION["num_flags"])
    for flag in VERIFICATION["flags"][:20]:
        print("  frame %-5d t=%-7s score=%.3f near_boundary=%s"
              % (flag["frame_index"], flag["timestamp_seconds"], flag["score"],
                 flag["near_track_boundary"]))
    if len(VERIFICATION["flags"]) > 20:
        print("  ... %d more" % (len(VERIFICATION["flags"]) - 20))

    DECISION, DECISION_REASONS = decision_mod.decide(PIPELINE_A, VERIFICATION)
    print("\nDECISION : %s" % DECISION)
    for reason in DECISION_REASONS:
        print("   - %s" % reason)
    if DECISION == decision_mod.SAFE:
        print("\nSAFE was earned: every check passed, none were assumed.")
    else:
        print("\nNEEDS_REVIEW is the expected default. SAFE is only earned.")

    check("gate: verifier ran over the rendered file",
          VERIFICATION["planned_frames"] > 0,
          "%s frames planned" % VERIFICATION["planned_frames"])
    check("gate: coverage complete", VERIFICATION["coverage"] >= 1.0,
          "coverage=%.4f" % VERIFICATION["coverage"])
    check("gate: no residual faces", VERIFICATION["num_flags"] == 0,
          "%d flags" % VERIFICATION["num_flags"])
    check("gate: produced a terminal verdict",
          DECISION in (decision_mod.SAFE, decision_mod.NEEDS_REVIEW,
                       decision_mod.FAILED), DECISION)

    hit_crops = [f for f in VERIFICATION["flags"]
                 if f.get("crop_path") and Path(f["crop_path"]).is_file()]
    if hit_crops:
        show = hit_crops[:6]
        fig, axes = plt.subplots(1, len(show), figsize=(2.6 * len(show), 3.0))
        axes = np.atleast_1d(axes)
        for ax, flag in zip(axes, show):
            crop = cv2.imread(flag["crop_path"])
            if crop is not None:
                ax.imshow(crop[:, :, ::-1])
            ax.set_title("f%d  %.2f" % (flag["frame_index"], flag["score"]),
                         fontsize=8)
            ax.axis("off")
        plt.suptitle("Faces RetinaFace still found in the ANONYMISED video",
                     fontsize=10)
        plt.tight_layout()
        plt.show()
        print("showing %d of %d hit crops" % (len(show), len(hit_crops)))
    else:
        print("no verifier hits -> no residual face crops to show")

## Side-by-side: original vs anonymised

Frames are grabbed by **sequential lockstep decode**, never by seeking, and only
at indices where both videos still have a frame. Frame picks prefer the
verifier's flagged frames when the pass-2 sweep was skipped
(`RUN_PASS2_SWEEP = False`), so the leak-prone cases are still shown.

In [ ]:
banner("CELL 8  side-by-side")


def read_at(path, indices):
    """Sequentially decode and grab the requested frames (seeking is unreliable)."""
    want = sorted(set(int(i) for i in indices))
    out = {}
    if not want:
        return out
    cap = cv2.VideoCapture(str(path))
    idx = 0
    while idx <= want[-1]:
        ok, frame = cap.read()
        if not ok or frame is None:
            break
        if idx in want:
            out[idx] = frame.copy()
        idx += 1
    cap.release()
    return out


# Prefer a bridged (inferred) frame - the leak-prone case is always shown.
bridged = [b for b in (PIPELINE_A.get("bridges") or []) if b.get("gap_frames")]
picks = []
if bridged:
    mid = (bridged[0]["first_bridged_frame"] + bridged[0]["last_bridged_frame"]) // 2
    picks.append(mid)
    print("showing bridged frame %s (gap=%s frames)"
          % (mid, bridged[0]["gap_frames"]))
else:
    print("no bridged gap in this run - showing frames with the most faces")
if picks:
    # Also show the single frame Pipeline A covered most densely.
    counts = ORIG_STATS["per_frame_counts"]
    if counts:
        picks.append(max(range(len(counts)), key=lambda i: counts[i]))
    if ANON_STATS is not None:
        picks.append(len(ANON_STATS["per_frame_counts"]) // 2)
else:
    busiest = sorted(range(len(ORIG_STATS["per_frame_counts"])),
                     key=lambda i: -ORIG_STATS["per_frame_counts"][i])[:2]
    picks.extend(busiest)
    if ANON_STATS is not None:
        picks.append(len(ANON_STATS["per_frame_counts"]) // 2)
# When the pass-2 sweep was skipped, prefer the verifier's flagged frames:
# those are the leak-prone cases the comparison must show.
if ANON_STATS is None and RUN_GATE:
    try:
        flagged = sorted({f["frame_index"] for f in VERIFICATION["flags"]})
        picks.extend(flagged[:3])
        if flagged:
            print("showing verifier-flagged frames %s" % flagged[:3])
    except NameError:
        pass
if ANON_STATS is None:
    picks.append(CLIP_FRAMES // 2)
denom = (len(ANON_STATS["per_frame_counts"])
         if ANON_STATS is not None else CLIP_FRAMES)
picks = sorted({p for p in picks
                if 0 <= p < min(len(ORIG_STATS["per_frame_counts"]),
                                denom)})[:3]

originals = read_at(CLIP_PATH, picks)
outputs = read_at(RENDERED, picks)
check("side-by-side: frames readable from both videos",
      bool(originals) and bool(outputs), "frames=%s" % picks)

if originals and outputs:
    fig, axes = plt.subplots(len(picks), 2, figsize=(13, 3.1 * len(picks)))
    axes = np.atleast_2d(axes)
    for row, idx in enumerate(picks):
        axes[row][0].imshow(originals[idx][:, :, ::-1])
        axes[row][0].set_title("original   f%d" % idx, fontsize=9)
        axes[row][1].imshow(outputs[idx][:, :, ::-1])
        axes[row][1].set_title("anonymised f%d" % idx, fontsize=9)
        for col in (0, 1):
            axes[row][col].axis("off")
    plt.tight_layout()
    plt.show()

## Detection statistics, both passes

**A drop is expected. A drop is not proof.** RetinaFace failing to parse a
pixelated face is weak evidence — it would equally fail on a badly-lit but
perfectly visible face. Treat the reduction as a sanity signal: if it does not
fall, something is wrong; if it falls, that alone clears nothing.

Cell 19 is where leaks are actually looked for. With `RUN_PASS2_SWEEP = False`
(default) the anonymised column below comes from that same gate sweep —
same file, same model family, same threshold, stricter stride-1 coverage.

In [ ]:
banner("CELL 9  detection statistics")

if ANON_STATS is None:
    # Pass-2 sweep skipped: build the anonymised side from the gate's
    # stride-1 sweep of the same file (same model family + threshold).
    try:
        gate_flags = VERIFICATION["flags"]
    except NameError:
        raise RuntimeError(
            "RUN_PASS2_SWEEP=False needs the gate to have run: execute the "
            "verification cell first, or set RUN_PASS2_SWEEP=True.")
    import collections as _collections

    _per = _collections.Counter(f["frame_index"] for f in gate_flags)
    _nframes = PIPELINE_A["rendered_frames"]
    ANON_STATS = {
        "label": "ANONYMISED (via gate run_verify)",
        "frames": _nframes,
        "fps": ORIG_STATS["fps"],
        "width": ORIG_STATS["width"],
        "height": ORIG_STATS["height"],
        "per_frame_counts": [_per.get(i, 0) for i in range(_nframes)],
        "detected_frames": _nframes,
        "held_frames": 0,
        "frames_with_faces": sum(1 for i in range(_nframes) if _per.get(i)),
        "total_detections": len(gate_flags),
        "max_in_frame": max(_per.values()) if _per else 0,
        "mean_per_frame": (len(gate_flags) / float(_nframes)) if _nframes else 0.0,
        "mean_score": (sum(f["score"] for f in gate_flags) / len(gate_flags))
                      if gate_flags else 0.0,
        "max_score": max((f["score"] for f in gate_flags), default=0.0),
        "scores": [f["score"] for f in gate_flags],
        "boxes": [f["box"] for f in gate_flags],
        "held_boxes": [],
        "elapsed_seconds": VERIFICATION.get("elapsed_seconds", 0.0),
    }
    print("anonymised side taken from the gate sweep (RUN_PASS2_SWEEP=False).\n")


def pct(part, whole):
    return (100.0 * part / whole) if whole else 0.0


rows = [
    ("frames", ORIG_STATS["frames"], ANON_STATS["frames"]),
    ("frames with >=1 face", ORIG_STATS["frames_with_faces"],
     ANON_STATS["frames_with_faces"]),
    ("  as % of frames", pct(ORIG_STATS["frames_with_faces"], ORIG_STATS["frames"]),
     pct(ANON_STATS["frames_with_faces"], ANON_STATS["frames"])),
    ("total detections", ORIG_STATS["total_detections"], ANON_STATS["total_detections"]),
    ("  as % of original", pct(ORIG_STATS["total_detections"], ORIG_STATS["total_detections"]),
     pct(ANON_STATS["total_detections"], ORIG_STATS["total_detections"])),
    ("max faces in one frame", ORIG_STATS["max_in_frame"], ANON_STATS["max_in_frame"]),
    ("mean faces/frame", ORIG_STATS["mean_per_frame"], ANON_STATS["mean_per_frame"]),
    ("mean score", ORIG_STATS["mean_score"], ANON_STATS["mean_score"]),
    ("max score", ORIG_STATS["max_score"], ANON_STATS["max_score"]),
    ("detection elapsed (s)", ORIG_STATS["elapsed_seconds"],
     ANON_STATS["elapsed_seconds"]),
]

print("%-26s %14s %14s %12s" % ("METRIC", "ORIGINAL", "ANONYMISED", "RETAINED"))
print("-" * 70)
for name, before, after in rows:
    if name == "  as % of frames" or name == "  as % of original":
        delta = "%6.1f%%" % (after - before)
    else:
        delta = "%6.1f%%" % pct(after, before)
    print("%-26s %14s %14s %12s"
          % (name, ("%.2f" % before) if isinstance(before, float) else before,
             ("%.2f" % after) if isinstance(after, float) else after, delta))

print("\nfaces still detected after anonymisation: %d over %d frames"
      % (ANON_STATS["total_detections"], ANON_STATS["frames"]))
if ANON_STATS["total_detections"] == 0:
    print("  -> RetinaFace finds nothing in the anonymised video. Expected, and")
    print("     NOT proof of anonymisation; see the caveat above.")
else:
    print("  -> Residual detections. These are the gate's flags - the leak")
    print("     candidates the verification cell reports in detail.")

# Distribution plot: how many faces per frame, both passes.
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
axes[0].plot(ORIG_STATS["per_frame_counts"], color="#2ca02c", lw=1.0,
             label="original")
axes[0].set_title("faces per frame - original", fontsize=10)
axes[1].plot(ANON_STATS["per_frame_counts"], color="#d62728", lw=1.0,
             label="anonymised")
axes[1].set_title("faces per frame - anonymised", fontsize=10)
for ax in axes:
    ax.set_xlabel("frame")
    ax.set_ylabel("count")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

if ORIG_STATS["scores"] and ANON_STATS["scores"]:
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.hist(ORIG_STATS["scores"], bins=20, alpha=0.6, color="#2ca02c",
            label="original")
    ax.hist(ANON_STATS["scores"], bins=20, alpha=0.6, color="#d62728",
            label="anonymised")
    ax.axvline(SCORE_THRESHOLD, color="k", ls="--", lw=1,
               label="threshold %.2f" % SCORE_THRESHOLD)
    ax.set_xlabel("detection score")
    ax.legend(fontsize=8)
    ax.set_title("RetinaFace score distribution", fontsize=10)
    plt.tight_layout()
    plt.show()

check("stats: anonymised detections did not increase",
      ANON_STATS["total_detections"] <= ORIG_STATS["total_detections"],
      "%d -> %d" % (ORIG_STATS["total_detections"], ANON_STATS["total_detections"]))
check("stats: baseline clip actually had faces to find",
      ORIG_STATS["total_detections"] > 0,
      "%d detections in the original" % ORIG_STATS["total_detections"])

## Summary

All checks from both passes, then every file this run produced.

In [ ]:
banner("CELL 11  summary")

width = max(len(step) for step, _, _ in RESULTS)
print("%-*s  %-6s %s" % (width, "STEP", "RESULT", "DETAIL"))
print("-" * (width + 40))
for step, status, detail in RESULTS:
    print("%-*s  %-6s %s" % (width, step, status, detail))

passed = sum(1 for _, status, _ in RESULTS if status == "PASS")
failed = [step for step, status, _ in RESULTS if status == "FAIL"]
print("\n%d/%d checks PASS" % (passed, len(RESULTS)))
if failed:
    print("FAILED: " + ", ".join(failed))
else:
    print("no failures")

print("\nheadline numbers")
print("  params      : stride %d, %s, pass-2 sweep %s"
      % (DET_STRIDE, ("detect width %dpx" % DET_WIDTH) if DET_WIDTH
         else "full resolution", "on" if RUN_PASS2_SWEEP else "off (gate stands in)"))
print("  original   : %d detections over %d frames (%d frames with a face)"
      % (ORIG_STATS["total_detections"], ORIG_STATS["frames"],
         ORIG_STATS["frames_with_faces"]))
print("  anonymised : %d detections over %d frames (%d frames with a face)"
      % (ANON_STATS["total_detections"], ANON_STATS["frames"],
         ANON_STATS["frames_with_faces"]))
print("  pipeline A : %d tracks, %d bridged gaps, %d frames covered"
      % (PIPELINE_A["num_tracks"], PIPELINE_A["num_bridged_gaps"],
         PIPELINE_A["covered_frames"]))
if RUN_GATE:
    print("  gate       : %s (%d verifier hits)"
          % (DECISION, VERIFICATION["num_flags"]))

print("\nfiles written:")
for path in [CLIP_PATH, ORIG_OVERLAY, ANON_OVERLAY, RENDERED]:
    if Path(path).is_file():
        print("  %-52s %9.1f KB"
              % (Path(path).name, Path(path).stat().st_size / 1024.0))